In [3]:
%%pyspark

# Lista todos os arquivos e diretórios dentro do container 'raw'
file_list = mssparkutils.fs.ls("abfss://raw@<storage_account>.dfs.core.windows.net/")

# Itera sobre cada arquivo para criar as Temp Views
for file_path in file_list:
    print(f"Lendo: {file_path.path}")
    
    # 1. Carrega o arquivo diretamente usando o caminho original
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(file_path.path)
    
    # 2. Trata o nome da view (remove 'SalesLT.' e '.csv')
    view_name = file_path.name.replace('SalesLT.', '').removesuffix('.csv')
    
    # 3. Cria a View Temporária para consultas SQL
    df.createOrReplaceTempView(view_name)
    print(f"View criada com sucesso: {view_name}")

StatementMeta(SparkPool01, 0, 4, Finished, Available, Finished, False)

Lendo: abfss://raw@<storage_account>.dfs.core.windows.net/SalesLT.Address.csv
View criada com sucesso: Address
Lendo: abfss://raw@<storage_account>.dfs.core.windows.net/SalesLT.Customer.csv
View criada com sucesso: Customer
Lendo: abfss://raw@<storage_account>.dfs.core.windows.net/SalesLT.CustomerAddress.csv
View criada com sucesso: CustomerAddress
Lendo: abfss://raw@<storage_account>.dfs.core.windows.net/SalesLT.Product.csv
View criada com sucesso: Product
Lendo: abfss://raw@<storage_account>.dfs.core.windows.net/SalesLT.ProductCategory.csv
View criada com sucesso: ProductCategory
Lendo: abfss://raw@<storage_account>.dfs.core.windows.net/SalesLT.ProductDescription.csv
View criada com sucesso: ProductDescription
Lendo: abfss://raw@<storage_account>.dfs.core.windows.net/SalesLT.ProductModel.csv
View criada com sucesso: ProductModel
Lendo: abfss://raw@<storage_account>.dfs.core.windows.net/SalesLT.ProductModelProductDescription.csv
View criada com sucesso: ProductModelProductDescription


In [4]:
views = spark.sql("SHOW VIEWS")
display(views)

StatementMeta(SparkPool01, 0, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e41c2f8b-0130-4c53-8780-0f72f63de11c)

In [3]:
df_sql = spark.sql("SELECT * FROM address LIMIT 100")
display(df_sql)


StatementMeta(SparkPool01, 8, 4, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 11db9167-7afa-49ad-8535-aea85590025b)

# Read Temp Views, clean the data, load into Spark Dataframes
- Filter Rows
- Rename Columns
- Drop Columns

In [5]:
from pyspark.sql.functions import rand, col, expr

df_salesorderdetail = spark.sql("SELECT SalesOrderID, OrderQty, ProductID,UnitPrice, UnitPriceDIscount, LineTotal FROM salesorderdetail")
df_salesorderheader = spark.sql("SELECT SalesOrderID, RevisionNumber, OrderDate, DueDate, ShipDate, Status, OnlineOrderFlag, SalesOrderNumber, PurchaseOrderNumber, AccountNumber, CustomerID, ShipToAddressID, BillToAddressID, ShipMethod, SubTotal, TaxAmt, Freight, TotalDue FROM salesorderheader")

# Randomizing the dates in the OrderDate column since our toy AdventureWorks LT dataset only has one distinct order date.

df_salesorderheader = df_salesorderheader.drop("OrderDate").withColumn("OrderDate", expr("date_add(current_date()-1000, CAST(rand() * 365 AS INT))"))

df_customer = spark.sql("SELECT CustomerID, Title, FirstName, MiddleName, LastName, Suffix, CompanyName, EmailAddress, Phone  FROM customer")
df_customeraddress = spark.sql("SELECT CustomerID, AddressID, AddressType FROM customeraddress")
df_product = spark.sql("SELECT ProductID, Name, ProductNumber, Color, StandardCost, ListPrice, Size, Weight, ProductCategoryID FROM product")
df_productcategory = spark.sql("SELECT ProductCategoryID, ParentProductCategoryID, Name FROM productcategory")

StatementMeta(SparkPool01, 0, 6, Finished, Available, Finished, False)

# Write Data Frames into Datalake Enriched layer as Tables


In [7]:
path="abfss://enriched@adlsposgraduacaodaniel.dfs.core.windows.net/"

tableName="salesOrderHeader"

df_salesorderheader.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(path +"/" + tableName)


StatementMeta(SparkPool01, 0, 8, Finished, Available, Finished, False)

In [8]:
path="abfss://enriched@adlsposgraduacaodaniel.dfs.core.windows.net/"

tableName="salesOrderDetail"

df_salesorderdetail.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(path + "/" + tableName)


StatementMeta(SparkPool01, 0, 9, Finished, Available, Finished, False)

In [9]:
path="abfss://enriched@adlsposgraduacaodaniel.dfs.core.windows.net/"

tableName="salesCustomer"

df_customer.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(path + "/" + tableName)

StatementMeta(SparkPool01, 0, 10, Finished, Available, Finished, False)

In [10]:
path="abfss://enriched@adlsposgraduacaodaniel.dfs.core.windows.net/"

tableName="salesCustomerAddress"

df_customeraddress.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(path + "/" + tableName)


StatementMeta(SparkPool01, 0, 11, Finished, Available, Finished, False)

In [11]:
path="abfss://enriched@adlsposgraduacaodaniel.dfs.core.windows.net/"

tableName="salesProduct"

df_product.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(path + "/" + tableName + "/")

StatementMeta(SparkPool01, 0, 12, Finished, Available, Finished, False)

In [12]:
path="abfss://enriched@adlsposgraduacaodaniel.dfs.core.windows.net/"

tableName="salesProductCategory"

df_productcategory.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(path + "/" + tableName)


StatementMeta(SparkPool01, 0, 13, Finished, Available, Finished, False)